In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 12

# 1. Read the data
df_base = pd.read_csv('baseline_and_ensemble_metrics.csv')
df_cross = pd.read_csv('cross_dataset_metrics.csv')

# --- FIGURE 1: IN-DOMAIN PERFORMANCE ---
# Find best F1 scores for each feature category per dataset
features_of_interest = ['tfidf_text', 'cv_text', 'lfs_cv']
feature_labels = ['TF-IDF', 'CountVec', 'LFS+CV', 'Tri-View Ensemble']

isot_scores, welfake_scores = [], []

for feat in features_of_interest:
    # Get max F1 for ISOT and WELFake for the current feature set
    isot_max = df_base[(df_base['dataset'] == 'ISOT') & (df_base['features'] == feat)]['f1'].max()
    welfake_max = df_base[(df_base['dataset'] == 'WELFake') & (df_base['features'] == feat)]['f1'].max()
    isot_scores.append(isot_max)
    welfake_scores.append(welfake_max)

# Get Tri-View Ensemble F1
isot_ens = df_base[(df_base['dataset'] == 'ISOT') & (df_base['model'] == 'TriViewEnsemble')]['f1'].max()
welfake_ens = df_base[(df_base['dataset'] == 'WELFake') & (df_base['model'] == 'TriViewEnsemble')]['f1'].max()
isot_scores.append(isot_ens)
welfake_scores.append(welfake_ens)

# Plot Figure 1
x = np.arange(len(feature_labels))
width = 0.35

fig1, ax1 = plt.subplots(figsize=(10, 6))
rects1 = ax1.bar(x - width/2, isot_scores, width, label='ISOT Dataset', color='#1f77b4')
rects2 = ax1.bar(x + width/2, welfake_scores, width, label='WELFake Dataset', color='#ff7f0e')

ax1.set_ylabel('F1-Score', fontweight='bold')
ax1.set_title('Comparison of Feature Views and Ensemble (In-Domain)', fontweight='bold', pad=15)
ax1.set_xticks(x)
ax1.set_xticklabels(feature_labels, fontweight='bold')
ax1.set_ylim(0.9, 1.01) # Set Y limit to highlight differences at the top end
ax1.legend(loc='lower right')

# Add values on top of bars
def autolabel(rects, ax):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

autolabel(rects1, ax1)
autolabel(rects2, ax1)

plt.tight_layout()
plt.savefig('figure1_indomain.png', dpi=300)
plt.close()


# --- FIGURE 3: DOMAIN SHIFT GAP ---
# Extract Source F1 vs Target F1 for the cross-dataset models
# Cross model 1: LogReg on lfs_cv (ISOT -> WELFake)
isot_source_f1 = df_base[(df_base['dataset'] == 'ISOT') & (df_base['model'] == 'LogReg (ISOT)') & (df_base['features'] == 'lfs_cv')]['f1'].max()
isot_target_f1 = df_cross[(df_cross['source_dataset'] == 'ISOT') & (df_cross['target_dataset'] == 'WELFake')]['f1'].values[0]

# Cross model 2: LinearSVC on tfidf_text (WELFake -> ISOT)
welfake_source_f1 = df_base[(df_base['dataset'] == 'WELFake') & (df_base['model'] == 'LinearSVC (WELFake)') & (df_base['features'] == 'tfidf_text')]['f1'].max()
welfake_target_f1 = df_cross[(df_cross['source_dataset'] == 'WELFake') & (df_cross['target_dataset'] == 'ISOT')]['f1'].values[0]

groups = ['Model Trained on ISOT\n(LogReg w/ LFS+CV)', 'Model Trained on WELFake\n(LinearSVC w/ TF-IDF)']
in_domain_f1 = [isot_source_f1, welfake_source_f1]
cross_domain_f1 = [isot_target_f1, welfake_target_f1]

x2 = np.arange(len(groups))

fig2, ax2 = plt.subplots(figsize=(8, 6))
rects3 = ax2.bar(x2 - width/2, in_domain_f1, width, label='Tested In-Domain (Source)', color='#2ca02c')
rects4 = ax2.bar(x2 + width/2, cross_domain_f1, width, label='Tested Cross-Domain (Target)', color='#d62728')

ax2.set_ylabel('F1-Score', fontweight='bold')
ax2.set_title('The Domain Shift Gap (Cross-Dataset Evaluation)', fontweight='bold', pad=15)
ax2.set_xticks(x2)
ax2.set_xticklabels(groups, fontweight='bold')
ax2.set_ylim(0, 1.1)
ax2.legend(loc="center right")

autolabel(rects3, ax2)
autolabel(rects4, ax2)

plt.tight_layout()
plt.savefig('figure3_crossdomain.png', dpi=300)
plt.close()